# AmazonQA-RAG — Phase 2: Retrieval Augmented Generation Pipeline

> **Prerequisite:** Phase 1 (`01_qlora_finetune.ipynb`) completed → LoRA adapter saved to `artifacts/lora_adapter/`.

---

## Why we add RAG on top of a fine-tuned model

Fine-tuning teaches the model **how to talk** about a domain (tone, format, common entities). It does **not** reliably teach it **what** is currently true — product catalogs change daily, models can't memorise SKUs, and parametric memory hallucinates.

**RAG fixes the "what":** at query time, we fetch the top-`k` most relevant chunks from a fresh vector index and inject them into the prompt. The fine-tuned model is now grounded in actual evidence instead of guessing from training-time recall.

Phase 3 will quantify this with **RAGAS faithfulness** — we expect a large jump going from `Base LLM` to `RAG+Base LLM`, and a further jump going to `RAG+Fine-tuned`.

## Pipeline

```
  Amazon QA CSV  ──┐
                   ▼
          [ RecursiveCharacterTextSplitter ]
                   │  chunk_size=512, overlap=50
                   ▼
          [ all-MiniLM-L6-v2 (384-d) ]
                   │  normalize → cosine
                   ▼
                  FAISS  (flat L2 index)
                   │  top-k=5 similarity search
                   ▼
    ┌──────────────┴──────────────┐
    │  RetrievalQA chain (stuff)  │
    │  RAG prompt template        │
    │  Mistral-7B + LoRA (Phase1) │
    └─────────────────────────────┘
                   │
                   ▼
         {answer, source_documents}
```

---
## 1. Install dependencies (RAG additions on top of Phase 1)

In [ ]:
%%capture
# Defensive NumPy pin — if you ran Phase 1 in this same runtime, NumPy 1.x
# is already pinned and this is a no-op. If you opened a fresh runtime for
# Phase 2 only, this prevents the dreaded 'numpy.dtype size changed' error.
!pip install -q "numpy<2"

!pip install -q -U \
    langchain==0.2.16 \
    langchain-community==0.2.16 \
    langchain-huggingface==0.0.3 \
    sentence-transformers==3.0.1 \
    faiss-cpu==1.8.0.post1

!pip install -q --force-reinstall "numpy<2"

# Phase 1 already installed: transformers, peft, bitsandbytes, accelerate, datasets

**Why `faiss-cpu` and not `faiss-gpu`?**  
Indexing our ~5k chunks takes ~2 seconds on CPU. The GPU build is finicky to install on Colab and offers no real benefit until corpora exceed ~1M vectors. We keep the GPU for the LLM.

---
## 2. Project setup

Same path-resolution dance as Phase 1 — point to the project root containing `configs/` and `src/`.

In [ ]:
import sys, os
PROJECT_ROOT = '/content/AmazonQA-RAG'
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# If you stored Phase 1's adapter on Drive, link it back into place:
from pathlib import Path
drive_adapter = Path('/content/drive/MyDrive/AmazonQA-RAG/lora_adapter')
local_adapter = Path(PROJECT_ROOT) / 'artifacts' / 'lora_adapter'
if drive_adapter.exists() and not local_adapter.exists():
    local_adapter.parent.mkdir(parents=True, exist_ok=True)
    os.system(f'cp -r {drive_adapter} {local_adapter}')
    print('Restored adapter from Drive.')

print('Adapter present:', local_adapter.exists())

In [ ]:
from configs.config import get_default_config, get_phi2_fallback_config

USE_PHI2_FALLBACK = False  # match whatever you used in Phase 1
cfg = get_phi2_fallback_config() if USE_PHI2_FALLBACK else get_default_config()

print(f'Base model       : {cfg.model.base_model_id}')
print(f'Embedding model  : {cfg.rag.embedding_model_id}')
print(f'Chunk size       : {cfg.rag.chunk_size} chars, overlap {cfg.rag.chunk_overlap}')
print(f'Top-k retrieval  : {cfg.rag.top_k}')
print(f'Vector store dir : {cfg.paths.vector_store_dir}')

---
## 3. Bootstrap a corpus

For demo purposes we re-use the same Amazon-QA dataset Phase 1 trained on, but treat it as a **knowledge base** rather than training data. In a real deployment you'd swap this for your own product catalog / FAQ exports — `cfg.rag.corpus_path` is the only thing that changes.

In [ ]:
from src.rag_pipeline import build_corpus_from_hf, load_documents

# Writes data/raw/amazon_corpus.csv
corpus_path = build_corpus_from_hf(cfg, output_format='csv', overwrite=False)
print('Corpus path:', corpus_path)

# Quick peek
import pandas as pd
pd.read_csv(corpus_path).head(3)

In [ ]:
docs = load_documents(cfg)
print(f'Loaded {len(docs)} documents')
print('\n--- example doc ---')
print(docs[0].page_content)
print('metadata:', docs[0].metadata)

---
## 4. Chunk the corpus

**RecursiveCharacterTextSplitter** tries separators in order — `"\n\n" → "\n" → ". " → " " → ""` — so it keeps paragraphs together when possible, falls back to sentences, then words. Naïve fixed-length splitting would cut mid-sentence and tank embedding quality.

In [ ]:
from src.rag_pipeline import chunk_documents

chunks = chunk_documents(docs, cfg)
print(f'{len(docs)} docs → {len(chunks)} chunks')

# Distribution check — should cluster near chunk_size, never exceed it
import numpy as np
lens = [len(c.page_content) for c in chunks]
print(f'chunk lengths  min={min(lens)}  median={int(np.median(lens))}  max={max(lens)}')

---
## 5. Build the FAISS index

We embed every chunk with **`sentence-transformers/all-MiniLM-L6-v2`** — 22M params, 384-dim output, fast enough to embed our entire corpus in seconds. Vectors are L2-normalised so cosine similarity reduces to a pure inner product, which is what FAISS's `IndexFlatIP` computes.

The resulting index is written to `artifacts/faiss_index/` so Phases 3 and 4 (and reruns of this notebook) can `load_vector_store(cfg)` instead of re-embedding.

In [ ]:
from src.rag_pipeline import build_vector_store

vs = build_vector_store(chunks, cfg, persist=True)
print(f'FAISS index built — {vs.index.ntotal} vectors of dim {vs.index.d}')
!ls -lh {cfg.paths.vector_store_dir}

---
## 6. Test retrieval ALONE (no LLM yet)

Before plugging in the expensive LLM, verify the retriever actually returns sensible chunks. If retrieval is bad, no amount of generation tuning will save us — garbage in, garbage out.

In [ ]:
from src.rag_pipeline import build_retriever

retriever = build_retriever(vs, cfg)

TEST_Q = 'Does this Bluetooth speaker work with iPhone?'
hits = retriever.invoke(TEST_Q)

print(f'Top-{len(hits)} chunks for: "{TEST_Q}"\n')
for i, doc in enumerate(hits, 1):
    snippet = doc.page_content[:200].replace('\n', ' ')
    print(f'[{i}] {snippet}...')
    print()

---
## 7. Load the fine-tuned LLM (Phase 1 adapter)

`load_llm(cfg, mode='auto')` automatically picks the fine-tuned model if Phase 1's adapter is present, otherwise falls back to the base. We'll explicitly request `'finetuned'` to assert the adapter is in play.

In [ ]:
from src.llm_loader import load_llm

llm = load_llm(cfg, mode='finetuned')

# Quick sanity check — generate without retrieval
raw_out = llm.invoke('What is the return policy for electronics on Amazon?')
print('Direct LLM output (no RAG):')
print(raw_out[:500])

---
## 8. Assemble the RAG chain

LangChain's `RetrievalQA` chain with `chain_type='stuff'` does the simplest, most effective thing: retrieve top-k chunks, concatenate them into the `{context}` slot of our prompt template, and ask the LLM to answer using only that context.

**Why `stuff` and not `map_reduce` / `refine`?**
- `stuff` — 1 LLM call, lowest latency, works perfectly when total chunks fit in context
- `map_reduce` — `k+1` LLM calls, used when chunks exceed context window (not our case)
- `refine` — `k` sequential LLM calls, expensive and tends to drift

5 chunks × ~500 chars ≈ 700 tokens of context — comfortably fits in Mistral's 8k window.

In [ ]:
from src.rag_pipeline import build_rag_chain, answer

rag_chain = build_rag_chain(llm, retriever, cfg)

# The same prompt as the sanity check — but now grounded
result = answer(rag_chain, 'What is the return policy for electronics on Amazon?')

print('=== ANSWER ===')
print(result['answer'])
print('\n=== SOURCES ===')
for i, src in enumerate(result['sources'], 1):
    print(f'[{i}] {src.page_content[:180]}...\n')

---
## 9. Qualitative comparison: Base vs. RAG+Base vs. RAG+Fine-tuned

A quick eyeball test before Phase 3 quantifies it with RAGAS. We expect:
- **Base LLM** — fluent but may invent specifics (hallucination)
- **RAG + Base** — grounded in retrieved evidence, less hallucinated
- **RAG + Fine-tuned** — grounded AND speaks in proper Amazon-QA tone/format

We need three LLM modes loaded. To stay under VRAM, we'll evaluate one at a time and free between.

In [ ]:
import gc, torch

QUERIES = [
    'Is the Echo Dot 5th gen waterproof?',
    'Does this USB-C cable support 100W fast charging?',
    'How long is the warranty on Kindle Paperwhite?',
]

def free_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

results_table = {q: {} for q in QUERIES}

In [ ]:
# (a) Base LLM only — no RAG, no fine-tuning
from src.llm_loader import load_llm

del llm, rag_chain
free_gpu()

base_llm = load_llm(cfg, mode='base')
for q in QUERIES:
    out = base_llm.invoke(cfg.prompt_template.format(question=q, answer='').rstrip())
    results_table[q]['base'] = out.strip()[:300]
    print(f'\nQ: {q}\nA (base): {results_table[q]["base"]}')
del base_llm
free_gpu()

In [ ]:
# (b) RAG + Base LLM
from src.rag_pipeline import build_rag_chain, build_retriever, load_vector_store, answer

vs = load_vector_store(cfg)
retriever = build_retriever(vs, cfg)
base_llm = load_llm(cfg, mode='base')
rag_base = build_rag_chain(base_llm, retriever, cfg)

for q in QUERIES:
    out = answer(rag_base, q)
    results_table[q]['rag_base'] = out['answer'][:300]
    print(f'\nQ: {q}\nA (rag+base): {results_table[q]["rag_base"]}')

del base_llm, rag_base
free_gpu()

In [ ]:
# (c) RAG + Fine-tuned LLM (our full system)
ft_llm = load_llm(cfg, mode='finetuned')
rag_ft = build_rag_chain(ft_llm, retriever, cfg)

for q in QUERIES:
    out = answer(rag_ft, q)
    results_table[q]['rag_ft'] = out['answer'][:300]
    print(f'\nQ: {q}\nA (rag+ft): {results_table[q]["rag_ft"]}')

In [ ]:
# Pretty side-by-side
import pandas as pd
df = pd.DataFrame(results_table).T
df.columns = ['Base LLM', 'RAG + Base', 'RAG + Fine-tuned']
df.index.name = 'Question'
df

---
## 10. Persist + handoff to Phase 3

Everything Phase 3 (RAGAS evaluation) needs is now on disk:
- `artifacts/lora_adapter/`  — fine-tuned weights (from Phase 1)
- `artifacts/faiss_index/`   — vector store (built above)
- `data/raw/amazon_corpus.csv` — source corpus

Drive-backup the FAISS index too if you're prone to losing Colab sessions:

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DRIVE_VS = '/content/drive/MyDrive/AmazonQA-RAG/faiss_index'
    !mkdir -p "{DRIVE_VS}"
    !cp -r "{cfg.paths.vector_store_dir}"/* "{DRIVE_VS}/"
    print(f'FAISS index persisted to: {DRIVE_VS}')
except Exception as e:
    print(f'(Skipping Drive backup: {e})')

---
## Phase 2 complete

**Deliverables:**
- `artifacts/faiss_index/` — 5k-chunk FAISS index over Amazon QA corpus
- A working `RetrievalQA` chain stacked on the QLoRA-fine-tuned Mistral-7B
- Side-by-side qualitative comparison across the three systems

**Next — Phase 3 will:**
1. Sample 100 held-out QA pairs as the evaluation set
2. Generate answers from all 3 systems for each question
3. Score every (question, answer, contexts, ground_truth) tuple with RAGAS
4. Output the ablation table + matplotlib bar chart that will headline the README

### Troubleshooting cheat-sheet

| Symptom | Fix |
|---|---|
| `FileNotFoundError: lora_adapter` | Run Phase 1 first, or copy adapter from Drive |
| FAISS retrieval returns garbage | Check that chunks aren't empty — re-run cell 4 with `print(chunks[0].page_content)` |
| LLM repeats the prompt | The `return_full_text=False` flag in `llm_loader.py` should prevent this — verify the pipeline created with that flag |
| OOM when loading second LLM | Always `del` and `free_gpu()` between model swaps (see cell 12) |
| `allow_dangerous_deserialization` error | Newer LangChain requires this flag for FAISS pickles — already set in `load_vector_store` |